## Color Blindness Simulation and Correction

This notebook provides functionality to simulate different types of color blindness (specifically deuteranopia) and to apply color correction (daltonization) to images for individuals with that specific type of color blindness. It includes functions to process single images and to batch process images within a folder.

In [ ]:
pip install daltonize

### Library Installations

First, we install the necessary Python libraries. `daltonize` is the core library used for color blind simulation and correction, while `setuptools` is a common dependency often used for package management.

In [ ]:
pip install setuptools

### Single Image Processing

This section defines functions to simulate deuteranopia and daltonize a single image. It also includes utility functions for color space transformations and converting NumPy arrays to PIL images.

read single img

In [ ]:
from PIL import Image
import numpy as np
import daltonize
from pkg_resources import parse_version


# Ensure numpy version compatibility
assert parse_version(np.__version__) >= parse_version('1.9.0'), "numpy >= 1.9.0 is required for daltonize"

def transform_colorspace(img, mat):
    """Transform image to a different color space."""
    return img @ mat.T

def simulate_deuteranopia(rgb):
    """Simulate the effect of deuteranopia on an image.

    Arguments:
    ----------
    rgb : array of shape (M, N, 3)
        original image in RGB format, values must be [0, 1] bounded

    Returns:
    --------
    sim_rgb : array of shape (M, N, 3)
        simulated image in RGB format
    """
    # Deuteranopia transformation matrix
    deuteranopia_matrix = np.array([[1, 0, 0], [1.10104433,  0, -0.00901975], [0, 0, 1]], dtype=np.float16)

    # RGB to LMS space transformation matrix
    rgb2lms = np.array([[0.3904725, 0.54990437, 0.00890159],
                        [0.07092586, 0.96310739, 0.00135809],
                        [0.02314268, 0.12801221, 0.93605194]], dtype=np.float16)

    # Precomputed inverse LMS to RGB matrix
    lms2rgb = np.array([[2.85831110e+00, -1.62870796e+00, -2.48186967e-02],
                        [-2.10434776e-01, 1.15841493e+00, 3.20463334e-04],
                        [-4.18895045e-02, -1.18154333e-01, 1.06888657e+00]], dtype=np.float16)

    # Transform from RGB to LMS space
    lms = transform_colorspace(rgb, rgb2lms)
    # Apply the deuteranopia matrix
    sim_lms = transform_colorspace(lms, deuteranopia_matrix)
    # Transform back to RGB
    sim_rgb = transform_colorspace(sim_lms, lms2rgb)

    return np.clip(sim_rgb, 0, 1)

def daltonize_deuteranopia(rgb):
    """Adjust color palette of an image to compensate for deuteranopia.

    Arguments:
    ----------
    rgb : array of shape (M, N, 3)
        original image in RGB format, values must be [0, 1] bounded

    Returns:
    --------
    dtpn : array of shape (M, N, 3)
        image in RGB format with colors adjusted
    """
    sim_rgb = simulate_deuteranopia(rgb)
    err2mod = np.array([[0, 0, 0], [0.7, 1, 0], [0.7, 0, 1]])

    # rgb - sim_rgb contains the color information that deuteranopes
    # cannot see. err2mod rotates this to a part of the spectrum that
    # they can see.
    err = transform_colorspace(rgb - sim_rgb, err2mod)
    dtpn = err + rgb
    return np.clip(dtpn, 0, 1)

def array_to_img(arr, gamma=2.4):
    """Convert a numpy array to a PIL image."""
    arr = np.clip(arr * 255, 0, 255).astype('uint8')
    img = Image.fromarray(arr, mode='RGB')
    return img

# Example usage
if __name__ == "__main__":
    img = Image.open("/content/sample_data/1a_frame_20.jpg")
    img_arr = np.array(img) / 255.0  # Normalize to [0, 1]

    # Simulate deuteranopia
    sim_img_arr = simulate_deuteranopia(img_arr)
    sim_img = array_to_img(sim_img_arr)
    sim_img.save("simulated_deuteranopia.png")

    # Daltonize to adjust for deuteranopia
    daltonized_img_arr = daltonize_deuteranopia(img_arr)
    daltonized_img = array_to_img(daltonized_img_arr)
    daltonized_img.save("daltonized_deuteranopia.png")


#### Function Definitions

*   `transform_colorspace(img, mat)`: A helper function to transform an image's color space using a given matrix.
*   `simulate_deuteranopia(rgb)`: Simulates the visual effect of deuteranopia (red-green color blindness) on an input RGB image. It converts the image to LMS color space, applies a specific transformation matrix for deuteranopia, and then converts it back to RGB.
*   `daltonize_deuteranopia(rgb)`: Adjusts the colors of an image to make it more discernible for individuals with deuteranopia. It calculates the color information lost due to deuteranopia and re-maps it to a visible spectrum.
*   `array_to_img(arr)`: Converts a normalized NumPy array (values 0-1) back into a Pillow (PIL) Image object, scaling values to 0-255.

read whole folder

### Batch Image Processing (Whole Folder)

This section extends the functionality to process multiple images stored in a directory. It reuses the core simulation and daltonization logic but wraps it in a function that iterates through all image files in a specified input folder.

In [ ]:
import os
from PIL import Image
import numpy as np
from pkg_resources import parse_version

# Ensure numpy version compatibility
assert parse_version(np.__version__) >= parse_version('1.9.0'), "numpy >= 1.9.0 is required for daltonize"

def transform_colorspace(img, mat):
    """Transform image to a different color space."""
    return img @ mat.T

def simulate_deuteranopia(rgb):
    """Simulate the effect of deuteranopia on an image."""
    # Deuteranopia transformation matrix
    deuteranopia_matrix = np.array([[1, 0, 0], [1.10104433,  0, -0.00901975], [0, 0, 1]], dtype=np.float16)

    # RGB to LMS space transformation matrix
    rgb2lms = np.array([[0.3904725, 0.54990437, 0.00890159],
                        [0.07092586, 0.96310739, 0.00135809],
                        [0.02314268, 0.12801221, 0.93605194]], dtype=np.float16)

    # Inverse LMS to RGB matrix
    lms2rgb = np.array([[2.85831110e+00, -1.62870796e+00, -2.48186967e-02],
                        [-2.10434776e-01, 1.15841493e+00, 3.20463334e-04],
                        [-4.18895045e-02, -1.18154333e-01, 1.06888657e+00]], dtype=np.float16)

    # Transform from RGB to LMS space
    lms = transform_colorspace(rgb, rgb2lms)
    # Apply the deuteranopia matrix
    sim_lms = transform_colorspace(lms, deuteranopia_matrix)
    # Transform back to RGB
    sim_rgb = transform_colorspace(sim_lms, lms2rgb)

    return np.clip(sim_rgb, 0, 1)

def daltonize_deuteranopia(rgb):
    """Adjust color palette of an image to compensate for deuteranopia."""
    sim_rgb = simulate_deuteranopia(rgb)
    err2mod = np.array([[0, 0, 0], [0.7, 1, 0], [0.7, 0, 1]])

    # Adjust colors
    err = transform_colorspace(rgb - sim_rgb, err2mod)
    dtpn = err + rgb
    return np.clip(dtpn, 0, 1)

def array_to_img(arr):
    """Convert a numpy array to a PIL image."""
    arr = np.clip(arr * 255, 0, 255).astype('uint8')
    img = Image.fromarray(arr, mode='RGB')
    return img

def process_folder(input_folder, output_folder_sim, output_folder_dalton):
    """Process all images in a folder for colorblind simulation and daltonization."""
    # Ensure output directories exist
    os.makedirs(output_folder_sim, exist_ok=True)
    os.makedirs(output_folder_dalton, exist_ok=True)

    # Loop through all image files in the input folder
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):  # Check for image files
            input_path = os.path.join(input_folder, filename)
            print(f"Processing {input_path}...")

            # Open image and normalize
            img = Image.open(input_path)
            img_arr = np.array(img) / 255.0  # Normalize to [0, 1]

            # Simulate deuteranopia
            sim_img_arr = simulate_deuteranopia(img_arr)
            sim_img = array_to_img(sim_img_arr)
            sim_output_path = os.path.join(output_folder_sim, f"simulated_{filename}")
            sim_img.save(sim_output_path)
            print(f"Saved simulated image: {sim_output_path}")

            # Daltonize for deuteranopia
            daltonized_img_arr = daltonize_deuteranopia(img_arr)
            daltonized_img = array_to_img(daltonized_img_arr)
            dalton_output_path = os.path.join(output_folder_dalton, f"daltonized_{filename}")
            daltonized_img.save(dalton_output_path)
            print(f"Saved daltonized image: {dalton_output_path}")

if __name__ == "__main__":
    # Input and output directories
    input_folder = "/content/drive/MyDrive/Loc/Notre dame"  # Folder containing images to process
    output_folder_sim = "/content/drive/MyDrive/Loc/Notre Dame Simulated"  # Folder to save simulated images
    output_folder_dalton = "/content/drive/MyDrive/Loc/Notre Dame Deuteranopia"  # Folder to save daltonized images

    # Process all images in the input folder
    process_folder(input_folder, output_folder_sim, output_folder_dalton)


#### Function Definitions and Usage

*   `process_folder(input_folder, output_folder_sim, output_folder_dalton)`: This function takes an `input_folder` containing original images and two output folders (`output_folder_sim` for simulated images and `output_folder_dalton` for daltonized images). It iterates through each image, applies both `simulate_deuteranopia` and `daltonize_deuteranopia` functions, and saves the results in their respective output directories.

The `if __name__ == "__main__":` block demonstrates how to use the `process_folder` function. It specifies the input folder (`/content/drive/MyDrive/Loc/Notre dame`) and the output folders where the processed images will be saved.

**Note**: This code assumes you have a Google Drive mounted and the specified paths are accessible. You may need to adjust `input_folder`, `output_folder_sim`, and `output_folder_dalton` to match your own directory structure.